In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import pandas as pd
from src.parsers import parse_stories


In [2]:
df=pd.read_csv("../data/interim/listings_parsed.csv")

In [3]:
df.shape

(183490, 22)

In [4]:
df.head()

,Listing Title,Area,Distances to Railway Stations,Building Age,Stories Tall,Floor,Price,Administrative Costs,Deposit,Gratuity,...,Links to Listing,jnc,Price_yen,Administrative Costs_yen,Deposit_yen,Gratuity_yen,Admin_yen,deposit_months,gratuity_months,lookalike_id
0,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000055105229/?bc=...,55105229,65000,3000.0,NaN,NaN,3000.0,NaN,NaN,72960
1,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000055105230/?bc=...,55105230,65000,3000.0,NaN,NaN,3000.0,NaN,NaN,72960
2,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.8万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000054648287/?bc=...,54648287,68000,3000.0,NaN,NaN,3000.0,NaN,NaN,72961
3,カーザ コリーナ,東京都青梅市野上町２,ＪＲ青梅線/河辺駅 歩15分,築5年,3階建,3階,8.5万円,4000円,8.5万円,4.25万円,...,https://suumo.jp/chintai/jnc_000055414464/?bc=...,55414464,85000,4000.0,85000.0,42500.0,4000.0,1.0,0.5,20237
4,京王相模原線 京王よみうりランド駅 2階建 築14年,東京都稲城市矢野口,"京王相模原線/京王よみうりランド駅 歩10分,ＪＲ南武線/矢野口駅 歩17分,京王相模原線/...",築14年,2階建,1階,8.1万円,3900円,-,-,...,https://suumo.jp/chintai/jnc_000037983769/?bc=...,37983769,81000,3900.0,NaN,NaN,3900.0,NaN,NaN,84797


In [5]:
df.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id'],
      dtype='str')

In [6]:
df[["stories_above", "stories_below"]] = df["Stories Tall"].apply(lambda t: pd.Series(parse_stories(t)))
print((df["stories_above"] == "FAILED").sum())

0


In [7]:
df[df["Stories Tall"] == "平屋"][["Price_yen", "Square Meters", "Rooms"]].describe()

,Price_yen
count,213.000000
mean,96704.225352
std,56725.687428
min,39000.000000
25%,65000.000000
50%,80000.000000
75%,100000.000000
max,350000.000000


In [8]:
df.loc[df["stories_above"] == "FAILED", "Stories Tall"].value_counts().head(10)

Series([], Name: count, dtype: int64)

In [9]:
df['Floor'].value_counts().head(50) 

Floor
1階       60974
2階       54627
3階       26766
4階       12926
5階        7333
6階        4580
7階        3177
8階        2439
1-2階      2054
9階        1765
10階       1546
-          997
11階        857
B1階        479
13階        454
12階        418
1-3階       270
14階        258
15階        180
21階        143
19階        105
2-3階       104
16階         88
23階         74
24階         64
30階         61
18階         55
25階         46
29階         46
B2階         45
3-4階        42
17階         42
20階         41
35階         38
1-1階        38
B1-1階       36
22階         28
4-5階        24
2-1階        21
28階         17
32階         17
B1-2階       16
51階         16
33階         14
1-階         12
2-0階        11
27階          8
1-4階         8
1-10階        8
34階          7
Name: count, dtype: int64

In [10]:
df["Floor"].str.replace(r"\d", "9", regex=True).value_counts()

Floor
9階        174587
99階         4652
9-9階        2617
-            997
B9階          529
B9-9階         55
9-階           27
9-99階          9
M9階            5
99-階           4
99-99階         3
99-9階          2
9B9階           2
B9-階           1
Name: count, dtype: int64

In [11]:
from src.parsers import parse_floor
for t in ["1階", "12階", "1-2階", "1-階", "B1階", "B1-1階", "-", "M1階", "1B1階"]:
    print(repr(t), parse_floor(t))

'1階' (1, 1, 0)
'12階' (12, 12, 0)
'1-2階' (1, 2, 1)
'1-階' (1, 1, 1)
'B1階' (-1, -1, 0)
'B1-1階' (-1, 1, 1)
'-' (None, None, None)
'M1階' ('FAILED', None, None)
'1B1階' ('FAILED', None, None)


In [12]:
df[["floor_min", "floor_max", "is_multi_floor"]] = df["Floor"].apply(lambda t: pd.Series(parse_floor(t)))
print((df["floor_min"] == "FAILED").sum())

7


In [13]:
df.loc[df["floor_min"] == "FAILED", "Floor"].value_counts()

Floor
M2階     5
1B2階    2
Name: count, dtype: int64

In [14]:
for c in ["floor_min", "floor_max", "is_multi_floor"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df["is_multi_floor"].value_counts())
print(df[["floor_min", "floor_max"]].describe())
print(df["floor_min"].isna().sum())

is_multi_floor
0.0    179768
1.0      2718
Name: count, dtype: int64
           floor_min      floor_max
count  182486.000000  182486.000000
mean        2.690672       2.706525
std         2.643514       2.637635
min        -4.000000      -4.000000
25%         1.000000       1.000000
50%         2.000000       2.000000
75%         3.000000       3.000000
max        51.000000      51.000000
1004


In [15]:
df["Square Meters"].str.replace(r"\d", "9", regex=True).value_counts()

Square Meters
99.99m     128518
99.9m       29293
99m         23058
999.99m      1412
9.99m         550
999.9m        249
999m          183
9.9m          123
9m            104
Name: count, dtype: int64

In [16]:
df['Square Meters'].value_counts().head(50) 

Square Meters
19.87m    2044
20m       2037
18m       1487
40m       1155
21m       1116
30m        904
17m        894
25m        868
16m        831
22m        732
35m        686
23m        682
19m        665
25.5m      662
15m        606
24m        605
20.28m     600
33m        587
19.44m     567
17.41m     562
19.8m      526
25.2m      517
26m        496
23.18m     475
16.5m      463
25.37m     459
26.08m     458
25.52m     454
38m        447
25.65m     438
25.6m      422
45m        421
36m        413
17.39m     413
28m        409
25.92m     408
42m        397
17.01m     396
25.38m     370
25.55m     357
27m        357
20.25m     353
25.11m     342
22.68m     337
50m        333
32m        331
22.35m     317
34m        316
38.88m     313
19.83m     309
Name: count, dtype: int64

In [17]:
from src.parsers import parse_area
for t in ["19.87m", "20m", "abc"]:
    print(repr(t), parse_area(t))

'19.87m' 19.87
'20m' 20.0
'abc' ('FAILED', None)


In [18]:
df['Area'] = df['Square Meters'].apply(parse_area)
print((df["Area"] == "FAILED").sum())

0


In [19]:
df['Area'].describe()

count    183490.000000
mean         34.212695
std          19.580601
min           1.000000
25%          21.120000
50%          26.900000
75%          42.050000
max         562.630000
Name: Area, dtype: float64

In [20]:
df.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id', 'stories_above', 'stories_below',
       'floor_min', 'floor_max', 'is_multi_floor'],
      dtype='str')

In [21]:
df.head()

,Listing Title,Area,Distances to Railway Stations,Building Age,Stories Tall,Floor,Price,Administrative Costs,Deposit,Gratuity,...,Gratuity_yen,Admin_yen,deposit_months,gratuity_months,lookalike_id,stories_above,stories_below,floor_min,floor_max,is_multi_floor
0,ルアナ,47.32,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,NaN,3000.0,NaN,NaN,72960,2,0,1.0,1.0,0.0
1,ルアナ,47.32,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,NaN,3000.0,NaN,NaN,72960,2,0,1.0,1.0,0.0
2,ルアナ,51.06,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.8万円,3000円,-,-,...,NaN,3000.0,NaN,NaN,72961,2,0,1.0,1.0,0.0
3,カーザ コリーナ,57.29,ＪＲ青梅線/河辺駅 歩15分,築5年,3階建,3階,8.5万円,4000円,8.5万円,4.25万円,...,42500.0,4000.0,1.0,0.5,20237,3,0,3.0,3.0,0.0
4,京王相模原線 京王よみうりランド駅 2階建 築14年,42.49,"京王相模原線/京王よみうりランド駅 歩10分,ＪＲ南武線/矢野口駅 歩17分,京王相模原線/...",築14年,2階建,1階,8.1万円,3900円,-,-,...,NaN,3900.0,NaN,NaN,84797,2,0,1.0,1.0,0.0


In [22]:
df['Building Age'].value_counts().head(50)

Building Age
新築      28963
築31年     6450
築32年     6231
築30年     5985
築29年     5750
築33年     5260
築28年     4997
築2年      4979
築14年     4944
築27年     4913
築3年      4897
築13年     4851
築15年     4457
築34年     4242
築17年     4208
築16年     4027
築26年     3805
築5年      3635
築12年     3557
築4年      3510
築35年     3431
築18年     3400
築11年     3149
築6年      2895
築22年     2735
築19年     2735
築25年     2728
築23年     2668
築36年     2628
築1年      2615
築21年     2580
築7年      2524
築20年     2476
築24年     2381
築9年      2233
築8年      2180
築10年     2139
築37年     1898
築40年     1512
築41年     1497
築38年     1441
築42年     1270
築39年     1137
築47年     1039
築43年     1037
築48年      907
築44年      894
築46年      750
築45年      722
築49年      688
Name: count, dtype: int64

In [23]:
# Number of buildings with 築1年
df[df['Building Age'] == "築1年"][["Price_yen", "Square Meters", "Rooms"]].describe()

,Price_yen
count,2615.000000
mean,129994.531549
std,62710.623745
min,42000.000000
25%,92500.000000
50%,115000.000000
75%,150000.000000
max,650000.000000


In [24]:
df.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id', 'stories_above', 'stories_below',
       'floor_min', 'floor_max', 'is_multi_floor'],
      dtype='str')

In [25]:
from src.parsers import parse_building_age
df['Building_Age_Parsed']=df['Building Age'].apply(parse_building_age)
print((df['Building_Age_Parsed'] == "FAILED").sum())

0


In [26]:
df['Building_Age_Parsed'].value_counts()

Building_Age_Parsed
(0, 1)     28963
(31, 0)     6450
(32, 0)     6231
(30, 0)     5985
(29, 0)     5750
           ...  
(82, 0)        2
(69, 0)        2
(70, 0)        2
(74, 0)        1
(77, 0)        1
Name: count, Length: 82, dtype: int64

In [27]:
df[["age_years", "is_new"]] = df["Building Age"].apply(lambda t: pd.Series(parse_building_age(t)))
print((df["age_years"] == "FAILED").sum())   # run BEFORE to_numeric
print(df["is_new"].isna().sum())             # should be 4 if so 



0
0


In [28]:
df["age_years"] = pd.to_numeric(df["age_years"], errors="coerce")
print(df["is_new"].value_counts())   

is_new
0    154527
1     28963
Name: count, dtype: int64


In [29]:
bad = df[df["is_new"].isna()]
for idx, row in bad.iterrows():
    print(idx, "|", row["Building Age"], "|", row["Stories Tall"], "|", row["Price"])

   - 4 rows failed the age parser (values: [list]). Rule: [what you chose]. Test added.

In [30]:
print(bad["Building Age"].map(repr).tolist())

[]


In [31]:
df["age_is_lower_bound"] = df["Building Age"].str.contains("以上").astype(int)
print(df["age_is_lower_bound"].sum())   # expect 4

4


In [32]:
df[["age_years", "is_new"]] = df["Building Age"].apply(lambda t: pd.Series(parse_building_age(t)))
print((df["age_years"] == "FAILED").sum())      # expect 0
df["age_years"] = pd.to_numeric(df["age_years"], errors="coerce")
print(df["is_new"].value_counts())              # expect 154,527 and 28,963, total 183,490
print(df["age_years"].max())                    # now 99
print(df["age_is_lower_bound"].sum())           # expect 4

0
is_new
0    154527
1     28963
Name: count, dtype: int64
99
4


In [33]:
df.to_csv("../data/interim/listings_parsed_v2.csv", index=False, encoding="utf-8-sig")

In [34]:
print((df["stories_above"] == "FAILED").sum())
print((df["Stories Tall"] == "平屋").sum())
print((df["floor_min"] == "FAILED").sum())

0
213
0


In [35]:
df["floor_min"].isna().sum()

np.int64(1004)

In [36]:
bad = df[df["stories_above"] == "FAILED"]
print(bad["Stories Tall"].value_counts())   # expect only 平屋, 213

Series([], Name: count, dtype: int64)


In [37]:
df[["stories_above", "stories_below"]] = df["Stories Tall"].apply(lambda t: pd.Series(parse_stories(t)))
print((df["stories_above"] == "FAILED").sum())   # expect 0
for c in ["stories_above", "stories_below"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
print(df["stories_above"].isna().sum())          # expect 0

0
0


In [38]:
# is_hiraya feature
df["is_hiraya"] = (df["Stories Tall"] == "平屋").astype(int)

In [39]:
df.to_csv("../data/interim/listings_parsed_v3.csv", index=False, encoding="utf-8-sig")

In [40]:
df["flag_floor_above_building"] = (df["floor_max"] > df["stories_above"]).astype(int)
print(df["flag_floor_above_building"].sum())

41


In [41]:
df.groupby("age_years")["Price_yen"].median().sort_values(ascending=False).head(10)

age_years
79    204000.0
1     115000.0
0     108000.0
2     105000.0
6      97000.0
3      96000.0
13     95000.0
72     95000.0
69     94500.0
7      94000.0
Name: Price_yen, dtype: float64

In [42]:
g = df[df["age_years"].isin([0, 1])].groupby("age_years")
print(g["Area"].median())
print((df["Price_yen"] / df["Area"]).groupby(df["age_years"]).median().head(4))

age_years
0    27.17
1    26.04
Name: Area, dtype: float64
age_years
0    3782.505910
1    4176.516942
2    3946.329913
3    3774.703557
dtype: float64


In [43]:
df["flag_area_small"] = (df["Area"] < 10).astype(int)
df["flag_area_large"] = (df["Area"] > 300).astype(int)
print(df["flag_area_small"].sum(), df["flag_area_large"].sum())

777 22


In [44]:
# Admin ¥1 and deposit/gratuity ¥100 rows. Count them and look at the raw text
print((df["Administrative Costs_yen"] < 100).sum(), (df["Deposit_yen"] < 1000).sum(), (df["Gratuity_yen"] < 1000).sum())

4 2 1


In [45]:
# Show the rows with these low values one by one
for index, row in df[df["Administrative Costs_yen"] < 100].iterrows():
    print(row[["Administrative Costs_yen", "Deposit_yen", "Gratuity_yen", "Price_yen", "Square Meters", "Rooms"]])

Administrative Costs_yen        1.0
Deposit_yen                     NaN
Gratuity_yen                83000.0
Price_yen                     83000
Square Meters                21.28m
Rooms                            1K
Name: 32234, dtype: object
Administrative Costs_yen         2.0
Deposit_yen                 640000.0
Gratuity_yen                640000.0
Price_yen                     320000
Square Meters                 55.63m
Rooms                           1LDK
Name: 52065, dtype: object
Administrative Costs_yen         1.0
Deposit_yen                 110000.0
Gratuity_yen                110000.0
Price_yen                     110000
Square Meters                 43.69m
Rooms                             3K
Name: 83203, dtype: object
Administrative Costs_yen        1.0
Deposit_yen                 57000.0
Gratuity_yen                57000.0
Price_yen                     57000
Square Meters                   20m
Rooms                         ワンルーム
Name: 94757, dtype: object


In [46]:
# Now for deposit 
for index, row in df[df["Deposit_yen"] < 1000].iterrows():
    print(row[["Administrative Costs_yen", "Deposit_yen", "Gratuity_yen", "Price_yen", "Square Meters", "Rooms"]])

Administrative Costs_yen      5000.0
Deposit_yen                    100.0
Gratuity_yen                140000.0
Price_yen                     140000
Square Meters                  53.3m
Rooms                            3DK
Name: 74878, dtype: object
Administrative Costs_yen      5000.0
Deposit_yen                    100.0
Gratuity_yen                140000.0
Price_yen                     140000
Square Meters                  53.3m
Rooms                            3DK
Name: 165717, dtype: object


In [47]:
# Now for gratuity
for index, row in df[df["Gratuity_yen"] < 1000].iterrows():
    print(row[["Administrative Costs_yen", "Deposit_yen", "Gratuity_yen", "Price_yen", "Square Meters", "Rooms"]])

Administrative Costs_yen    10000.0
Deposit_yen                 83000.0
Gratuity_yen                  100.0
Price_yen                     83000
Square Meters                21.63m
Rooms                         ワンルーム
Name: 102093, dtype: object


In [48]:
df["flag_fee_tiny"] = ((df["Administrative Costs_yen"] < 100) |
                       (df["Deposit_yen"] < 1000) |
                       (df["Gratuity_yen"] < 1000)).astype(int)
print(df["flag_fee_tiny"].sum())

7


In [49]:
# Claude review 2
print(df.nlargest(5, "Price_yen")[["Rooms", "Area", "Price_yen"]])

        Rooms    Area  Price_yen
110728  4SLDK  365.51    4450000
175102   5LDK  327.94    3450000
160223   4LDK  414.63    3300000
117339   2LDK  256.40    3200000
30030    4LDK  217.99    3020000


In [50]:
## Claude review 1
df["no_gratuity"] = df["Gratuity_yen"].isna()
print(df.groupby("no_gratuity")["Price_yen"].median())

no_gratuity
False    91000.0
True     69000.0
Name: Price_yen, dtype: float64


In [51]:
df["Rooms"].str.replace(r"\d", "9", regex=True).value_counts()

Rooms
9K        78874
9LDK      42675
9DK       30371
ワンルーム     28417
9SLDK      2421
9SK         376
9SDK        292
9LK          51
9SLK         11
99SLDK        1
99K           1
Name: count, dtype: int64

In [52]:
## Oddities

odd = df[df["Rooms"].str.fullmatch(r"\d{2}\w*")]
print(odd[["Rooms", "Area", "Price_yen"]])

         Rooms  Area  Price_yen
97938   12SLDK  38.5     110000
115205     13K  25.0      73000


In [53]:
from src.parsers import parse_rooms
cols = ["n_rooms", "has_L", "has_D", "has_K", "has_S", "is_1R"]
df[cols] = df["Rooms"].apply(lambda t: pd.Series(parse_rooms(t)))
print((df["n_rooms"] == "FAILED").sum())   # aim for 0

0


In [54]:
df["area_per_room"] = df["Area"] / df["n_rooms"]
df["flag_rooms_implausible"] = (df["area_per_room"] < 6).astype(int)
print(df["flag_rooms_implausible"].sum())
print(df[df["flag_rooms_implausible"] == 1][["Rooms", "Area", "Price_yen"]].head(15))

47
       Rooms  Area  Price_yen
6561   ワンルーム  5.58      27000
6562   ワンルーム  5.58      29700
7883      1K  1.00      55000
13743  ワンルーム  3.31      29000
13744  ワンルーム  3.31      29000
22894  ワンルーム  4.95      45000
24803  ワンルーム  5.82      45800
24804  ワンルーム  5.82      45800
24805  ワンルーム  5.82      45800
24806  ワンルーム  5.82      48800
24807  ワンルーム  5.82      48800
35685   2LDK  5.10      60000
37517  ワンルーム  3.00      40000
37518  ワンルーム  3.00      40000
45583   3LDK  6.50      65000


In [55]:
# impossible: multi-room layout in a tiny area
df["flag_layout_area_conflict"] = ((df["n_rooms"] >= 2) & (df["Area"] < 15)).astype(int)
# tiny studio: keep as a separate, softer flag (already covered by flag_area_small if < 10 m²)

In [56]:
print(df["flag_layout_area_conflict"].sum())
print(df[df["flag_layout_area_conflict"] == 1][["Rooms", "Area", "Price_yen"]])

7
       Rooms   Area  Price_yen
35685   2LDK   5.10      60000
45583   3LDK   6.50      65000
94564     2K  14.58      40000
123159    2K  14.00      27000
133182    2K  14.00      34000
175807   6DK   8.56     190000
175808   6DK   8.56     190000


In [57]:
s = df["Distances to Railway Stations"]
print(s.str.count(",").value_counts().sort_index())        # how many stations per listing
print(s.str.contains("バス").sum())                          # bus entries
print(s.str.contains("車").sum())                            # car entries
print(s.sample(10, random_state=1).tolist())

Distances to Railway Stations
0      8071
1     11588
2    163831
Name: count, dtype: int64
14070
2289
['京王線/調布駅 歩4分,京王線/布田駅 歩5分,京王線/国領駅 歩13分', 'ＪＲ山手線/目黒駅 歩5分,東京メトロ南北線/目黒駅 歩5分,東急目黒線/不動前駅 歩12分', '東京メトロ半蔵門線/水天宮前駅 歩7分,都営新宿線/浜町駅 歩8分,都営大江戸線/清澄白河駅 歩10分', '西武新宿線/花小金井駅 歩18分,ＪＲ中央線/武蔵小金井駅 バス23分 (バス停)共済住宅前 歩1分,西武新宿線/小平駅 バス24分 (バス停)共済住宅前 歩1分', '東急田園都市線/三軒茶屋駅 歩11分,東急世田谷線/西太子堂駅 歩6分,東急世田谷線/若林駅 歩8分', 'ＪＲ総武線/新小岩駅 歩7分,ＪＲ総武線/平井駅 歩32分,ＪＲ総武線/小岩駅 歩34分', '京王線/府中駅 歩27分,京王線/東府中駅 歩31分,ＪＲ武蔵野線/北府中駅 歩29分', 'ＪＲ総武線/亀戸駅 歩9分,東武亀戸線/亀戸水神駅 歩2分,都営新宿線/大島駅 歩15分', '京急本線/梅屋敷駅 歩4分,京急本線/大森町駅 歩10分,京急本線/京急蒲田駅 歩15分', '西武池袋線/石神井公園駅 歩7分,西武池袋線/練馬高野台駅 歩13分,西武新宿線/上井草駅 歩24分']


In [58]:
bus = s[s.str.contains("バス")]
print(bus.sample(5, random_state=0).tolist())
car = s[s.str.contains("車")]
print(car.sample(5, random_state=0).tolist())

['ＪＲ横浜線/町田駅 バス15分 (バス停)藤の台団地 歩1分,小田急線/鶴川駅 バス18分 (バス停)藤の台団地 歩1分,ＪＲ横浜線/古淵駅 バス15分 (バス停)藤の台団地 歩1分', '西武新宿線/東伏見駅 歩3分,ＪＲ中央線/三鷹駅 バス23分 (バス停)東伏見駅 歩3分,西武新宿線/武蔵関駅 歩12分', '京王相模原線/京王永山駅 歩8分,小田急多摩線/小田急永山駅 歩9分,京王線/聖蹟桜ヶ丘駅 バス12分 (バス停)諏訪下橋 歩2分', '小田急線/成城学園前駅 歩9分,東急田園都市線/二子玉川駅 バス20分 (バス停)東京都市大学付属中高前 歩1分,小田急線/祖師ヶ谷大蔵駅 歩20分', '東京メトロ南北線/王子神谷駅 歩13分,ＪＲ京浜東北線/王子駅 バス12分 (バス停)環七新田 歩1分,ＪＲ埼京線/赤羽駅 バス15分 (バス停)環七新田 歩1分']
['東京メトロ千代田線/北綾瀬駅 歩25分,つくばエクスプレス/六町駅 歩26分,ＪＲ常磐線/亀有駅 車14分(4.1km)', '京王相模原線/京王堀之内駅 歩5分,京王相模原線/南大沢駅 車9分(3.5km),京王相模原線/多摩境駅 車11分(4.8km)', '小田急線/町田駅 歩9分,小田急小田原線/相模大野駅 歩29分,ＪＲ横浜線/古淵駅 車12分(2.7km)', 'ＪＲ中央線/西荻窪駅 歩13分,ＪＲ中央線/荻窪駅 バス5分 (バス停)日産自動車前 歩1分,西武新宿線/上井草駅 バス5分 (バス停)荻窪警察前 歩5分', '西武新宿線/花小金井駅 歩10分,西武新宿線/小平駅 車10分(2.8km),西武新宿線/田無駅 車10分(3.0km)']


In [59]:
# Markdown pending
from src.parsers import parse_station_entry

rows = []
for idx, text in df["Distances to Railway Stations"].items():
    for rank, entry in enumerate(str(text).split(",")):
        r = parse_station_entry(entry)
        if r is not None:
            rows.append({"listing_idx": idx, "rank": rank, **r})

stations = pd.DataFrame(rows)
print(len(stations), stations["listing_idx"].nunique())   # about 550k rows, 183,490 listings
print(stations["mode"].value_counts())

522728 183490
mode
walk    502023
bus      18085
car       2620
Name: count, dtype: int64


In [60]:
# Markdown pending
walk = stations[stations["mode"] == "walk"].sort_values(["listing_idx", "walk_min"])
g = walk.groupby("listing_idx")

feat = pd.DataFrame({
    "walk_min_nearest": g["walk_min"].min(),
    "walk_min_second": g["walk_min"].apply(lambda s: s.iloc[1] if len(s) > 1 else None),
    "n_walk_stations": g.size(),
    "n_lines": g["line"].nunique(),
})
feat["walk_le_5"] = (feat["walk_min_nearest"] <= 5).astype(int)
feat["walk_le_10"] = (feat["walk_min_nearest"] <= 10).astype(int)
feat["walk_le_15"] = (feat["walk_min_nearest"] <= 15).astype(int)

modes = stations.groupby("listing_idx")["mode"].agg(
    has_bus_access=lambda s: int((s == "bus").any()),
    has_car_access=lambda s: int((s == "car").any()),
)

df = df.join(feat).join(modes)

In [61]:
# Markdown pending
print(df["walk_min_nearest"].describe())
print(df.groupby("walk_min_nearest")["Price_yen"].median().head(15))

count    181825.000000
mean          7.981944
std           5.196404
min           1.000000
25%           4.000000
50%           7.000000
75%          10.000000
max         202.000000
Name: walk_min_nearest, dtype: float64
walk_min_nearest
1.0     100000.0
2.0      95000.0
3.0      92000.0
4.0      92000.0
5.0      87000.0
6.0      88000.0
7.0      83200.0
8.0      80000.0
9.0      85000.0
10.0     76000.0
11.0     78500.0
12.0     72000.0
13.0     72000.0
14.0     72000.0
15.0     69000.0
Name: Price_yen, dtype: float64


In [62]:
df.to_csv("../data/interim/listings_parsed_v4.csv", index=False, encoding="utf-8-sig")
stations.to_csv("../data/interim/stations_long.csv", index=False, encoding="utf-8-sig")

In [63]:
df.shape

(183490, 55)

In [64]:
df=pd.read_csv("../data/interim/listings_parsed_v4.csv")

In [69]:
print((df["walk_min_nearest"] > 30).sum(), (df["walk_min_nearest"] > 60).sum())
print(df.nlargest(5, "walk_min_nearest")[["Distances to Railway Stations", "Price_yen"]])

474 4
                            Distances to Railway Stations  Price_yen
105403  東急田園都市線/用賀駅 歩202分,東急田園都市線/桜新町駅 バス15分 (バス停)聖ドミニ...     135000
131088  ＪＲ中央線/八王子駅 バス25分 (バス停)新川口 歩8分,ＪＲ中央線/西八王子駅 歩68分...      78000
159227                                    ＪＲ中央線/八王子駅 歩68分      85000
159228                                    ＪＲ中央線/八王子駅 歩68分      85000
140734  小田急線/鶴川駅 バス10分 (バス停)野津田車庫 歩9分,小田急線/町田駅 バス18分 (...      76000


In [65]:
df.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id', 'stories_above', 'stories_below',
       'floor_min', 'floor_max', 'is_multi_floor', 'Building_Age_Parsed',
       'age_years', 'is_new', 'age_is_lower_bound', 'is_hiraya',
       'flag_floor_above_building', 'flag_area_small', 'flag_area_large',
       'flag_fee_tiny', 'no_gratuity', 'n_rooms', 'has_L', 'has_D', 'has_K',
       'has_S', 'is_1R', 'area_per_room', 'flag_rooms_implausible',
       'flag_layout_area_conflict', 'walk_min_nearest', 'walk_min_second',
       'n_walk_stations', 'n_lines', 'walk_le_5', 'walk_le_10', 'walk_le_15',
       'has_bus_access', 'has_car_access'],
      dtype='str')

In [66]:
df.head()

,Listing Title,Area,Distances to Railway Stations,Building Age,Stories Tall,Floor,Price,Administrative Costs,Deposit,Gratuity,...,flag_layout_area_conflict,walk_min_nearest,walk_min_second,n_walk_stations,n_lines,walk_le_5,walk_le_10,walk_le_15,has_bus_access,has_car_access
0,ルアナ,47.32,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,0,4.0,24.0,2.0,1.0,1.0,1.0,1.0,0,0
1,ルアナ,47.32,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,0,4.0,24.0,2.0,1.0,1.0,1.0,1.0,0,0
2,ルアナ,51.06,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.8万円,3000円,-,-,...,0,4.0,24.0,2.0,1.0,1.0,1.0,1.0,0,0
3,カーザ コリーナ,57.29,ＪＲ青梅線/河辺駅 歩15分,築5年,3階建,3階,8.5万円,4000円,8.5万円,4.25万円,...,0,15.0,NaN,1.0,1.0,0.0,0.0,1.0,0,0
4,京王相模原線 京王よみうりランド駅 2階建 築14年,42.49,"京王相模原線/京王よみうりランド駅 歩10分,ＪＲ南武線/矢野口駅 歩17分,京王相模原線/...",築14年,2階建,1階,8.1万円,3900円,-,-,...,0,10.0,15.0,3.0,2.0,0.0,1.0,1.0,0,0


In [70]:
print(df["Area"].dtype)
print(df["Area"].head(3).tolist())

float64
[47.32, 47.32, 51.06]


In [71]:
df = df.rename(columns={"Area": "area_m2"})

In [72]:
orig = pd.read_csv("../data/interim/listings_dedup.csv", dtype={"jnc": str}, usecols=["jnc", "Area"])
print(len(orig), len(df))   # both should be 183,490

183490 183490


In [73]:
print(orig["jnc"].is_unique, df["jnc"].is_unique)   # expect True, True

True True


In [75]:
for frame in (df, orig):
    frame["jnc"] = (
        frame["jnc"]
        .astype("string")   
        .str.strip()
        .str.zfill(12)
    )

if not orig["jnc"].is_unique:
    raise ValueError("orig contains duplicate jnc values")

df = df.drop(columns="location", errors="ignore")

location_by_jnc = orig.set_index("jnc")["Area"]
df["location"] = df["jnc"].map(location_by_jnc)

print(df["location"].isna().sum())
print(len(df))


0
183490


In [76]:
print(df[["Listing Title", "location", "area_m2"]].sample(5, random_state=0))

             Listing Title     location  area_m2
105540               カサベルデ   東京都杉並区西荻北５    22.03
43024   東急多摩川線 鵜の木駅 9階建 新築   東京都大田区鵜の木２    25.25
178151            ヴィレッタ・エヌ    東京都江東区亀戸４    26.57
24374             第二アルレ堀之内     東京都北区豊島２    26.54
121735          クリオ上野毛ラモード  東京都世田谷区上野毛１    22.59


In [77]:
print(df.index.equals(pd.RangeIndex(len(df))))

True


In [79]:
df.to_csv("../data/interim/listings_parsed_v5.csv", index=False, encoding="utf-8-sig")

In [80]:
df=pd.read_csv("../data/interim/listings_parsed_v5.csv")

In [82]:
print(df["location"].str.extract(r"^(東京都.{1,4}?[市区町村郡])")[0].value_counts().head(70))
print(df["location"].sample(15, random_state=2).tolist())

0
東京都世田谷区     9026
東京都練馬区      7751
東京都江戸川区     7227
東京都杉並区      7196
東京都板橋区      6907
東京都足立区      6756
東京都大田区      6551
東京都八王子市     6439
東京都新宿区      6254
東京都江東区      6012
東京都品川区      5908
東京都中野区      5836
東京都葛飾区      5015
東京都港区       5005
東京都町田市      4914
東京都豊島区      4910
東京都渋谷区      4544
東京都墨田区      4519
東京都目黒区      4465
東京都台東区      4407
東京都北区       4322
東京都府中市      4061
東京都中央区      3858
東京都文京区      3577
東京都三鷹市      3329
東京都立川市      3313
東京都調布市      3106
東京都日野市      2982
東京都小平市      2857
東京都西東京市     2788
東京都国分寺市     2746
東京都武蔵野市     2734
東京都多摩市      2572
東京都小金井市     2482
東京都荒川区      2352
東京都国立市      1889
東京都東村       1877
東京都昭島市      1660
東京都千代田区     1455
東京都青梅市      1255
東京都稲城市      1228
東京都東久留米市    1123
東京都狛江市      1100
東京都清瀬市      1004
東京都東大和市      879
東京都福生市       837
東京都羽村        797
東京都あきる野市     689
東京都武蔵村       659
東京都西多摩郡      317
Name: count, dtype: int64
['東京都中野区中央２', '東京都杉並区荻窪４', '東京都大田区鵜の木３', '東京都新宿区住吉町', '東京都東久留米市下里７', '東京都足立区入谷２', '東京都狛江市東和泉２', '東京都西東京市東町３', '東京都渋谷区幡ヶ谷３'